[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/koebe/koebe.ipynb)

# Koebe Polyhedra

This example computes the Koebe polyhedron of a triangulation of the sphere: every triangulation of the sphere is the pattern of a convex polyhedron whose edges all touch the unit sphere. Its polar dual touches the sphere at the same points, each dual edge crossing its edge there at right angles, which makes the pair the best-shaped mesh and dual there is: every dual vertex inside its face, and every dual edge positive. The polyhedron is fixed only up to the maps that keep the sphere, which centre it and carry it through a family of shapes.

The algebra is modelled with the unit sphere as its metric, x, y and z squaring to one and w to minus one. A circle on the sphere is a single vector, `n + s * w` scaled to square to one, and a point is an antivector, on the sphere when it is null and on a circle when `circle & point == 0`. A circle's dual, `circle.dual()`, is the apex of the cone that touches the sphere along it: polarity in the sphere is the dual. Two circles touch from outside exactly when `circle | other == -1`, and the sum of their poles, `circle.dual() + other.dual()`, is then the point where they touch, the null point on the edge between the two apexes. A face's circle is the dual of its vertices' wedge, `(a ^ b ^ c).dual()`, and the maps that keep the sphere are the rotors, which keep every inner product, so a boosted pattern still touches along every edge.

In the notation of circle packings the vertex circles are a packing whose tangency pattern is the triangulation, and the face circles its dual packing, which the Koebe–Andreev–Thurston theorem says exists and is unique up to Möbius transformations of the sphere.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.sparse import SparseExtensor
# The algebra x+y+z+w-, with the unit sphere as its metric, shared with the figures.
from examples.surfaces.koebe.core import ga, Mesh, hull, spiral
from examples.util.animation import save_figure, show_animation
from examples.surfaces.koebe import render

mv = NumpyContext(ga).multivector
# Circles on the sphere, the planes that cut it, are vectors.
Vector = ga.gatype.vector()                         # Vector+
# A point, of space or of the sphere, is an antivector, the pole of its plane; so is a direction, a point at infinity.
Point = ga.gatype.antivector()                      # Antivector-
# The centre of the sphere is the pole of the plane at infinity, w, which pairs with a point to give its weight.
origin = mv.w.dual()                                # [] Point-


## 1. An irregular triangulation

The triangulation is the convex hull of points at random distances along directions spread over the sphere, a Fibonacci spiral nudged a little at random, so that its triangles have uneven shapes.

In [ ]:
# The number of points, how far their directions are turned off an even spread, and its seed; the spread and
# seed of the random distances whose hull gives the triangulation, every point on it.
n_points = 32
jitter = 0.03
direction_seed = 0
mesh_spread, mesh_seed = 0.06, 57

In [ ]:
# Directions along a Fibonacci spiral, each turned a little at random, and the hull of the points at random
# distances along them: a triangulation of the sphere with triangles of uneven shape.
nudge_weights = np.random.default_rng(direction_seed).normal(size=(n_points, 3))   # [points, 3]
nudge_planes = (stack((mv.yz, mv.zx, mv.xy)) * nudge_weights).sum(axis=-1)     # [points] Bivector+
nudges = (nudge_planes * (jitter / 2)).exp()                                   # [points] Rotor+
directions = nudges >> spiral(n_points)                                        # [points] Point-
faces = hull(directions * np.exp(mesh_spread * np.random.default_rng(mesh_seed).normal(size=n_points)))   # [faces, 3]
mesh = Mesh(n_points, faces)
edges = mesh.edges                                                             # [edges, 2]


## 2. Circles that touch along every edge

The circle where the plane `n | x == s` cuts the sphere is the vector `n + s * w`, scaled to square to one, and a point lies on it exactly when `circle & point` is zero. Two circles touch from outside exactly when their inner product is minus one. A circle's dual is a point, the apex of the cone that touches the sphere along it, so the duals of circles touching along every edge are the vertices of a polyhedron whose every edge touches the sphere where its two circles do. The circle of a face, through its three touching points, is orthogonal to its vertices' circles, the dual of their wedge, and its apex is a vertex of the dual polyhedron. Gauss–Newton finds the circles from caps of one radius around the triangulation's directions, half the mean angle across an edge, each cap the polar of the apex of its cone. The conditions fix the circles only up to the six dimensions of maps that keep the sphere, along which the steps do not move.

In [ ]:
# Gauss-Newton steps from the caps to the touching circles.
n_newton_steps = 6


In [ ]:
def koebe(start: Vector, edges: np.ndarray, n_steps: int) -> Vector:
    """The circles that square to one and touch from outside along every edge, inner product minus one, follow
    by Gauss-Newton from the start."""
    tails, heads = edges[:, 0], edges[:, 1]
    n_vertices, n_edges = start.shape[-1], len(edges)
    # Each vertex's residual depends on its own circle, each edge's on the circles at its two ends.
    rows = np.concatenate([np.arange(n_vertices), n_vertices + np.arange(n_edges), n_vertices + np.arange(n_edges)])
    columns = np.concatenate([np.arange(n_vertices), tails, heads])
    circles = start                                                            # [vertices] Vector+
    for _ in range(n_steps):
        residual = concatenate([circles.squared() - 1, (circles[tails] | circles[heads]) + 1]).field()   # Scalar+[vertices + edges]
        # How the residuals change with the circles: a map from the circles to the residuals.
        jacobian = SparseExtensor.from_indices(
            concatenate([2 * circles | Vector, circles[heads] | Vector, circles[tails] | Vector]),
            rows, columns, (n_vertices + n_edges, n_vertices))                 # (Scalar+ <- Vector+)[vertices + edges <- vertices]
        # The change that zeroes the linearised residuals, along the circles' own metric: the step commutes with
        # the maps that keep the sphere, so a boosted start gives the boosted circles.
        circles = circles - jacobian.adjoint()(jacobian(jacobian.adjoint()).solve(residual)).batch()
    return circles


# Caps around the directions, half as wide as the mean angle across an edge, the size of the turn from one
# end's direction to the other's.
angles = (directions[edges[:, 1]] * directions[edges[:, 0]]).log().norm()      # [edges] Scalar+
radius = angles.mean() / 2                                                     # [] Scalar+
# Each cap is the polar of the apex of the cone touching the sphere along it, out along its direction at the
# reciprocal of the cosine, scaled so that the circle squares to one.
cap_apexes = (directions + origin * radius.cos()) / radius.sin()               # [points] Point-
start = cap_apexes.dual()                                                      # [points] Vector+
circles = koebe(start, edges, n_newton_steps)                                  # [points] Vector+

start_touch = start[edges[:, 0]] | start[edges[:, 1]]                          # [edges] Scalar+
print(f"inner products across the edges at the start, from {start_touch.to_array().min():.2f} to {start_touch.to_array().max():.2f}")


## 3. The maps that keep the sphere

The circles are fixed only up to the maps that keep the sphere, and those are the rotors of this algebra: rotations, and boosts, the rotors of a plane through w, which crowd the points of the sphere towards one direction, shrinking the circles there and growing those opposite. Every rotor keeps every inner product, so every boosted pattern still touches along every edge and every boosted polyhedron is still a Koebe polyhedron. The centred one has its touching points balanced about the centre of the sphere: the point inside the sphere, squaring to minus one, that minimises the summed logarithms of minus its inner products with them is the centre from which they balance, and the rotor carrying that point onto `origin`, the centre of the sphere, centres the pattern. The animation carries the centred pattern through boosts whose plane through w turns once about the z axis.

In [ ]:
# Fixed-point iterations for the centre.
n_centre_iterations = 8
# The boosts the centred polyhedron is carried through: their largest rapidity, the angle by which their plane
# through w is tilted from x towards the z axis it turns about, the frames of one turn, and the duration of each.
rapidity = 0.6
tilt = 0.4
n_frames = 96
duration_ms = 40

In [ ]:
def touching(circles: Vector, edges: np.ndarray) -> Point:
    """The point where the circles at the ends of each edge touch is the sum of their poles, the null point on
    the line between the apexes."""
    poles = circles.dual()                                                     # [...] Point-
    return poles[..., edges[:, 0]] + poles[..., edges[:, 1]]                   # [..., edges] Point-


# At the balancing centre the touching points, each weighted by minus the reciprocal of its inner product with
# the centre, sum along the centre itself: the centre is the fixed point of that weighted sum, normalized.
touches = touching(circles, edges)                                             # [edges] Point-
centre = origin                                                                # [] Point-
for _ in range(n_centre_iterations):
    weighted_sum = (touches * (-1 / (centre | touches))).sum()                 # [] Point-
    centre = weighted_sum.normalized()
# The boost along the line through both centres carries the balancing centre onto the centre of the sphere, and
# centres the pattern.
centring = (1 - origin * centre).normalized()                                  # [] Rotor+
centred_circles = centring >> circles                                          # [points] Vector+
save_figure(render.draw(centred_circles, faces, edges), "koebe")

turn_angles = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)              # [frames]
# The plane of the boost, through w, is tilted from x towards the z axis and turns once about it.
tilted_plane = (mv.zx * (tilt / 2)).exp() >> mv.xw                             # [] Bivector+
boost_planes = (mv.xy * (-turn_angles / 2)).exp() >> tilted_plane              # [frames] Bivector+
boosts = (boost_planes * (rapidity * np.sin(turn_angles) ** 2 / 2)).exp()      # [frames] Rotor+
boosted_circles = boosts[:, None] >> centred_circles                           # [frames, points] Vector+
show_animation(render.animate(boosted_circles, faces, edges), "koebe_boost", duration_ms)


In [ ]:
# checks
# The circles touch from outside along every edge once solved.
assert np.abs((circles[edges[:, 0]] | circles[edges[:, 1]]).to_array() + 1).max() < 1e-10
# Every circle squares to one, and the circles at the ends of every edge touch from outside, under every boost.
assert np.abs((boosted_circles | boosted_circles).to_array() - 1).max() < 1e-10
assert np.abs((boosted_circles[:, edges[:, 0]] | boosted_circles[:, edges[:, 1]]).to_array() + 1).max() < 1e-10


def centre_distances(apexes: Point, edges: np.ndarray) -> np.ndarray:
    """The distance from the centre to the line through the apexes at the ends of each edge is the line's part
    paired with w, its moment about the centre, over its part wedged with w, its direction."""
    lines = apexes[..., edges[:, 0]] & apexes[..., edges[:, 1]]                # [..., edges] Bivector-
    return ((lines | mv.w).squared() / (lines ^ mv.w).squared()).square_root().to_array()


# Every edge of the polyhedron and of its dual touches the sphere: its line's distance from the centre is one.
# The apexes are the circles' duals at unit weight along w; a face's circle is the dual of its vertices' wedge.
poles = boosted_circles.dual()                                                 # [frames, points] Point-
face_circles = (boosted_circles[..., faces[:, 0]] ^ boosted_circles[..., faces[:, 1]] ^ boosted_circles[..., faces[:, 2]]).dual()   # [frames, faces] Vector-
face_poles = face_circles.dual()                                               # [frames, faces] Point-
assert np.abs(centre_distances(poles / (mv.w & poles), edges) - 1).max() < 1e-10
assert np.abs(centre_distances(face_poles / (mv.w & face_poles), mesh.edge_faces) - 1).max() < 1e-10
# Centred, the touching points balance about the centre of the sphere: at unit weight they average to a point
# on the planes x, y and z through it.
centred_touches = touching(centred_circles, edges)                             # [edges] Point-
balance = (centred_touches / (mv.w & centred_touches)).mean()                  # [] Point-
assert np.abs((stack((mv.x, mv.y, mv.z)) & balance).to_array()).max() < 1e-4
